In [ ]:
!pip install fastapi uvicorn nest-asyncio python-multipart transformers torch pillow


In [1]:
yaml_config =  """
               model_path: "Salesforce/blip-vqa-base"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

build model

In [4]:
from transformers import BlipProcessor, BlipForQuestionAnswering
from omegaconf import OmegaConf
import torch
from PIL import Image

class VisualQuestionAnswering:
    def __init__(self, config_path):
        # Đọc file cấu hình config.yaml
        self.config = OmegaConf.load(config_path)

        # Load Processor và Model chuẩn cho bài toán Hỏi Đáp (VQA)
        self.processor = BlipProcessor.from_pretrained(self.config.model_path)
        self.model = BlipForQuestionAnswering.from_pretrained(self.config.model_path)

    def __call__(self, image: Image.Image, query: str):
        # Đưa cả ảnh và câu hỏi (query) vào processor
        inputs = self.processor(image, query, return_tensors="pt")

        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=50)

        return self.processor.decode(out[0], skip_special_tokens=True)

init model

In [5]:
# Khởi tạo instance và truyền đúng đường dẫn file config
vqa_model = VisualQuestionAnswering("config.yaml")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


init api

In [6]:
import torch
import nest_asyncio
import uvicorn
import threading
import io
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware

from PIL import Image

# 2. Khởi tạo FastAPI và thiết lập chặn chéo CORS
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# 3. Định nghĩa các endpoints
@app.get("/")
async def root():
    return {"message": "API Sinh mo ta hinh anh"}



@app.get("/health")
async def health_check():
    return {"status": "He thong hoat dong binh thuong"}

@app.post('/predict')
async def predict(file: UploadFile = File(...), query: str = Form(default="")):
    try:
        # Đọc file ảnh tải lên
        contents = await file.read()
        image = Image.open(io.BytesIO(contents)).convert("RGB")

        # Chạy model với thông số ảnh và query kết hợp
        result = vqa_model(image, query=query)

        # Trả về theo định dạng {"answer": ...} khớp với index.html
        return {
            "answer": result
        }
    except Exception as e:
        return {"error": str(e)}




run server

In [7]:
# 4. Sử dụng Threading để chạy server ngầm
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

nest_asyncio.apply()
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server đang chạy tải tại http://0.0.0.0:8000 (Local)")

Server đang chạy tải tại http://0.0.0.0:8000 (Local)


In [8]:
!ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io


ssh: Could not resolve hostname a.pinggy.io: Temporary failure in name resolution


In [11]:
# Cài đặt Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Chạy ngầm đường hầm hướng vào port 8000 và lấy link public
!nohup cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &
!sleep 5
!grep -o 'https://.*\.trycloudflare.com' cloudflared.log

(Reading database ... 118198 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
https://adjacent-documentary-tools-asset.trycloudflare.com
